In [ ]:
import os, sys
notebook_dir = os.path.dirname(os.path.abspath("__file__")) if '__file__' in globals() else os.getcwd()
cds_root = os.path.abspath(os.path.join(notebook_dir, ".."))
if cds_root not in sys.path:
    sys.path.insert(0, cds_root)
src_path = os.path.join(cds_root, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset as TorchDataset
import torchvision
from torchvision import transforms
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Dinov2Model,
    Wav2Vec2Processor,
    Wav2Vec2Model,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from datasets import Dataset
from src.data.image.data_loader import ImageDataset
import librosa

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_metadata_csv  = r"C:\Users\User\Downloads\cds\CDS\processed_test_metadata.csv"
train_metadata_csv = r"C:\Users\User\Downloads\cds\CDS\processed_train_metadata.csv"
checkpoint_path    = r"C:\Users\User\Downloads\cds\CDS\text_checkpoint\checkpoint-2500"
image_checkpoint_path = r"C:\Users\User\Downloads\cds\CDS\best_dinov2_emotion.pt"
audio_checkpoint_path = r"C:\Users\User\Downloads\cds\CDS\checkpoint.pt"
frames_root_dir    = r"C:\Users\User\Downloads\cds\CDS\processed_test_frames\\"
audio_dir          = r"C:\Users\User\Downloads\cds\CDS\processed_test_audio"
train_frames_root  = r"C:\Users\User\Downloads\cds\CDS\processed_train_frames\\"
train_audio_root   = r"C:\Users\User\Downloads\cds\CDS\processed_train_audio"

In [ ]:
# =====================
# Load test metadata
# =====================
test_df = pd.read_csv(test_metadata_csv)[["video_id", "utterance", "emotion"]]
label_list = sorted(test_df["emotion"].unique())
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
test_df["label"] = test_df["emotion"].map(label2id)

def clean_utterance(text):
    import re
    from unidecode import unidecode
    text = str(text)
    text = text.replace('…', '...').replace('\xa0', ' ')
    text = unidecode(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

video_to_text = {
    row["video_id"]: clean_utterance(row["utterance"])
    for _, row in test_df.iterrows()
}

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

# =====================
# Text Model
# =====================
tokenizer = AutoTokenizer.from_pretrained("bhadresh-savani/bert-base-uncased-emotion")
text_model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)
text_model = text_model.to(device)
text_model.eval()

# =====================
# Image Model
# =====================
class DINOv2EmotionClassifier(nn.Module):
    def __init__(self, num_labels, model_name="facebook/dinov2-base"):
        super().__init__()
        self.backbone = Dinov2Model.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, num_labels),
        )
        self.freeze_backbone()

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values).last_hidden_state
        cls_token = outputs[:, 0, :]
        return self.classifier(cls_token)

image_model = DINOv2EmotionClassifier(num_labels=len(label_list)).to(device)
image_ckpt = torch.load(image_checkpoint_path, map_location=device, weights_only=False)
image_model.load_state_dict(image_ckpt, strict=False)
image_model.eval()

# =====================
# Audio Model
# =====================
class Wav2Vec2EmotionClassifier(nn.Module):
    def __init__(self, num_classes, model_name="facebook/wav2vec2-base"):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(model_name)
        self.freeze_encoder()
        hidden_size = self.wav2vec2.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def freeze_encoder(self):
        for p in self.wav2vec2.parameters():
            p.requires_grad = False

    def forward(self, input_values):
        hidden = self.wav2vec2(input_values).last_hidden_state
        pooled = hidden.mean(dim=1)
        return self.classifier(pooled)

audio_ckpt = torch.load(audio_checkpoint_path, map_location=device, weights_only=False)
audio_label_list = [str(x) for x in audio_ckpt["label_classes"]]
audio_label2id = {label: idx for idx, label in enumerate(audio_label_list)}
processor = Wav2Vec2Processor.from_pretrained(audio_ckpt["model_name"])

audio_model = Wav2Vec2EmotionClassifier(
    num_classes=len(audio_label_list),
    model_name=audio_ckpt["model_name"],
).to(device)
audio_model.load_state_dict(audio_ckpt["model_state_dict"])
audio_model.eval()

In [ ]:
sys.path.append(r"C:\Users\User\Downloads\cds\CDS")
from src.utils import load_video_data

# =====================
# Dataset Classes
# =====================
fusion_name_map = {
    "anger":    "angry",
    "fear":     "fearful",
    "joy":      "happy",
    "sadness":  "sad",
    "surprise": "surprised",
}

class RawAudioDataset(TorchDataset):
    def __init__(self, df, audio_dir, label2id, name_map=None, sample_rate=16000):
        self.video_ids   = df["video_id"].tolist()
        self.emotions    = df["emotion"].tolist()
        self.audio_dir   = audio_dir
        self.label2id    = label2id
        self.name_map    = name_map or {}
        self.sample_rate = sample_rate

    def __len__(self):
        return len(self.video_ids)

    def __getitem__(self, idx):
        audio_path = os.path.join(self.audio_dir, self.video_ids[idx] + ".wav")
        waveform, sr = librosa.load(audio_path, sr=self.sample_rate, mono=True)
        if len(waveform) == 0:
            waveform = np.zeros(self.sample_rate, dtype=np.float32)
        emotion = self.name_map.get(self.emotions[idx], self.emotions[idx])
        return waveform.astype(np.float32), self.label2id[emotion]

class AudioCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, batch):
        waveforms, labels = zip(*batch)
        inputs = self.processor(
            list(waveforms),
            sampling_rate=16000,
            return_tensors="pt",
            padding=True,
        )
        return inputs.input_values.to(device), torch.LongTensor(labels).to(device)

audio_to_fusion_idx = {
    audio_label2id[fusion_name_map.get(label, label)]: fusion_idx
    for label, fusion_idx in label2id.items()
    if fusion_name_map.get(label, label) in audio_label2id
}

# =====================
# Test Data
# =====================
test_video_dirs_all, test_labels_all, _ = load_video_data(
    metadata_csv=test_metadata_csv,
    frames_root_dir=frames_root_dir
)

test_texts  = []
test_labels = []
test_video_dirs = []

for video_dir, label in zip(test_video_dirs_all, test_labels_all):
    video_id = os.path.basename(video_dir)
    if video_id not in video_to_text:
        continue
    if not os.path.exists(video_dir) or len(os.listdir(video_dir)) == 0:
        continue
    test_texts.append(video_to_text[video_id])
    test_labels.append(label)
    test_video_dirs.append(video_dir)

test_video_ids = [os.path.basename(d) for d in test_video_dirs]
test_id_order  = {vid: i for i, vid in enumerate(test_video_ids)}

image_dataset = ImageDataset(
    video_dirs=test_video_dirs,
    labels=test_labels,
    transform=transform,
    n_frames=8
)
image_loader = DataLoader(image_dataset, batch_size=8, shuffle=False, num_workers=0)

test_df_audio = pd.read_csv(test_metadata_csv)[["video_id", "emotion"]]
test_df_audio = test_df_audio[test_df_audio["video_id"].isin(test_video_ids)]
test_df_audio = test_df_audio.sort_values(
    "video_id", key=lambda col: col.map(test_id_order)
).reset_index(drop=True)

audio_dataset = RawAudioDataset(test_df_audio, audio_dir, audio_label2id, name_map=fusion_name_map)
audio_loader  = DataLoader(
    audio_dataset, batch_size=8, shuffle=False,
    collate_fn=AudioCollator(processor), num_workers=0,
)

# =====================
# Train Data
# =====================
train_df_meta = pd.read_csv(train_metadata_csv)[["video_id", "utterance", "emotion"]]
train_video_to_text = {
    row["video_id"]: clean_utterance(row["utterance"])
    for _, row in train_df_meta.iterrows()
}

train_video_dirs_all, train_labels_all, _ = load_video_data(
    metadata_csv=train_metadata_csv,
    frames_root_dir=train_frames_root
)

train_texts      = []
train_labels     = []
train_video_dirs = []

for video_dir, label in zip(train_video_dirs_all, train_labels_all):
    video_id = os.path.basename(video_dir)
    if video_id not in train_video_to_text:
        continue
    if not os.path.exists(video_dir) or len(os.listdir(video_dir)) == 0:
        continue
    train_texts.append(train_video_to_text[video_id])
    train_labels.append(label)
    train_video_dirs.append(video_dir)

train_video_ids = [os.path.basename(d) for d in train_video_dirs]
train_id_order  = {vid: i for i, vid in enumerate(train_video_ids)}

train_image_dataset = ImageDataset(
    video_dirs=train_video_dirs,
    labels=train_labels,
    transform=transform,
    n_frames=8
)
train_image_loader = DataLoader(train_image_dataset, batch_size=8, shuffle=False, num_workers=0)

train_df_audio = pd.read_csv(train_metadata_csv)[["video_id", "emotion"]]
train_df_audio = train_df_audio[train_df_audio["video_id"].isin(train_video_ids)]
train_df_audio = train_df_audio.sort_values(
    "video_id", key=lambda col: col.map(train_id_order)
).reset_index(drop=True)

train_audio_dataset = RawAudioDataset(train_df_audio, train_audio_root, audio_label2id, name_map=fusion_name_map)

print(f"Train: {len(train_texts)} samples")
print(f"Test:  {len(test_texts)} samples")

In [ ]:
import gc

# =====================
# Helper Functions
# =====================
def get_text_probs(model, tokenizer, texts, device):
    model.eval()
    probs_list = []
    with torch.no_grad():
        for text in texts:
            inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=128)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)
            probs_list.append(probs.cpu())
    return torch.cat(probs_list, dim=0)

def get_image_probs(model, loader, device):
    model.eval()
    probs_list = []
    with torch.no_grad():
        for imgs, _ in loader:
            b, t, c, h, w = imgs.shape
            imgs = imgs.view(b * t, c, h, w).to(device)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1)
            probs = probs.view(b, t, -1).mean(dim=1)
            probs_list.append(probs.cpu())
    return torch.cat(probs_list, dim=0)

def get_audio_probs_gpu(model, dataset, remap, device, max_seconds=5, batch_size=8):
    model.eval()
    model.to(device)
    max_samples = max_seconds * 16000
    probs_list = []
    with torch.no_grad():
        for i in range(0, len(dataset), batch_size):
            batch = [dataset[j] for j in range(i, min(i + batch_size, len(dataset)))]
            waveforms, _ = zip(*batch)
            waveforms = [w[:max_samples] for w in waveforms]
            inputs = processor(list(waveforms), sampling_rate=16000, return_tensors="pt", padding=True)
            outputs = model(inputs.input_values.to(device))
            probs = torch.softmax(outputs, dim=1)
            reordered = torch.zeros((probs.size(0), len(label_list)), device=device)
            for audio_idx, fusion_idx in remap.items():
                reordered[:, fusion_idx] = probs[:, audio_idx]
            probs_list.append(reordered.cpu())
    return torch.cat(probs_list, dim=0)

# =====================
# Extract Probabilities
# =====================
text_probs_train = get_text_probs(text_model, tokenizer, train_texts, device)
text_probs       = get_text_probs(text_model, tokenizer, test_texts, device)
text_model.cpu(); gc.collect(); torch.cuda.empty_cache()

image_probs_train = get_image_probs(image_model, train_image_loader, device)
image_probs       = get_image_probs(image_model, image_loader, device)
image_model.cpu(); gc.collect(); torch.cuda.empty_cache()

audio_probs_train = get_audio_probs_gpu(audio_model, train_audio_dataset, audio_to_fusion_idx, device)
audio_probs       = get_audio_probs_gpu(audio_model, audio_dataset, audio_to_fusion_idx, device)
audio_model.cpu(); gc.collect(); torch.cuda.empty_cache()

# =====================
# Single Modality Baselines
# =====================
test_labels_arr  = np.array(test_labels)
train_labels_arr = np.array(train_labels)

for name, probs in [("Text only", text_probs), ("Image only", image_probs), ("Audio only", audio_probs)]:
    preds = probs.numpy().argmax(axis=1)
    print(f"=== {name} ===")
    print("Accuracy:", accuracy_score(test_labels_arr, preds))
    print("Weighted F1:", f1_score(test_labels_arr, preds, average="weighted"))
    print()

In [ ]:
# =====================
# Late Fusion Model (reused by both fusion cells)
# =====================
class LateFusionLinear(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.fc(x)

# =====================
# Fusion 1: Image + Audio
# =====================
train_ia = np.concatenate([image_probs_train.numpy(), audio_probs_train.numpy()], axis=1)
test_ia  = np.concatenate([image_probs.numpy(),       audio_probs.numpy()],       axis=1)

train_ia_t = torch.tensor(train_ia, dtype=torch.float32).to(device)
test_ia_t  = torch.tensor(test_ia,  dtype=torch.float32).to(device)
train_y_t  = torch.tensor(train_labels_arr, dtype=torch.long).to(device)

fusion_ia_model = LateFusionLinear(14, len(label_list)).to(device)
optimizer_ia    = torch.optim.Adam(fusion_ia_model.parameters(), lr=1e-2)
criterion       = nn.CrossEntropyLoss()

for epoch in range(200):
    fusion_ia_model.train()
    optimizer_ia.zero_grad()
    criterion(fusion_ia_model(train_ia_t), train_y_t).backward()
    optimizer_ia.step()

fusion_ia_model.eval()
with torch.no_grad():
    test_preds_ia = fusion_ia_model(test_ia_t).argmax(dim=1).cpu().numpy()

torch.save(fusion_ia_model.state_dict(),
           r"C:\Users\User\Downloads\cds\CDS\late_fusion_ia.pt")

print("=== Image + Audio Fusion ===")
print("Accuracy:  ", accuracy_score(test_labels_arr, test_preds_ia))
print("Weighted F1:", f1_score(test_labels_arr, test_preds_ia, average="weighted"))
print("\nClassification Report:\n", classification_report(test_labels_arr, test_preds_ia, target_names=label_list, zero_division=0))
print("\nConfusion Matrix:\n", confusion_matrix(test_labels_arr, test_preds_ia))

In [ ]:
# =====================
# Fusion 2: Text + Image + Audio
# =====================
train_iat = np.concatenate([text_probs_train.numpy(), image_probs_train.numpy(), audio_probs_train.numpy()], axis=1)
test_iat  = np.concatenate([text_probs.numpy(),       image_probs.numpy(),       audio_probs.numpy()],       axis=1)

train_iat_t = torch.tensor(train_iat, dtype=torch.float32).to(device)
test_iat_t  = torch.tensor(test_iat,  dtype=torch.float32).to(device)

fusion_iat_model = LateFusionLinear(21, len(label_list)).to(device)
optimizer_iat    = torch.optim.Adam(fusion_iat_model.parameters(), lr=1e-2)

for epoch in range(200):
    fusion_iat_model.train()
    optimizer_iat.zero_grad()
    criterion(fusion_iat_model(train_iat_t), train_y_t).backward()
    optimizer_iat.step()

fusion_iat_model.eval()
with torch.no_grad():
    test_preds = fusion_iat_model(test_iat_t).argmax(dim=1).cpu().numpy()

torch.save(fusion_iat_model.state_dict(),
           r"C:\Users\User\Downloads\cds\CDS\late_fusion_model.pt")

print("=== Text + Image + Audio Fusion ===")
print("Accuracy:   ", accuracy_score(test_labels_arr, test_preds))
print("Weighted F1:", f1_score(test_labels_arr, test_preds, average="weighted"))
print("\nClassification Report:\n", classification_report(test_labels_arr, test_preds, target_names=label_list, zero_division=0))
print("\nConfusion Matrix:\n", confusion_matrix(test_labels_arr, test_preds))